# XGBoost Model Selection

This notebook evaluates an XGBoost classifier for the binary diabetes target using 5-fold stratified cross-validation. Metrics are computed directly from each validation fold.

In [1]:
import pandas as pd
import numpy as np
from sklearn.base import clone
from sklearn.metrics import recall_score, precision_score, f1_score, fbeta_score, average_precision_score, roc_auc_score, roc_curve, precision_recall_curve
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBClassifier

In [2]:
# ==========================================
# 1. Load Training Data ONLY
# ==========================================
train_df = pd.read_csv('../data/processed/train.csv')

X_train = train_df.drop(columns=['Diabetes_01'])
y_train = train_df['Diabetes_01']

In [3]:
# ==========================================
# 2. Setup Stratified Cross-Validation
# ==========================================
cv_strategy = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [4]:
# ==========================================
# 3. Build XGBoost Model
# ==========================================
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()
scale_pos_weight = negative_count / positive_count

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    scale_pos_weight=scale_pos_weight,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

In [5]:
# ==========================================
# 4. Run 5-Fold Cross-Validation and Collect Out-of-Fold Predictions
# ==========================================

fold_predictions = []

for fold_number, (train_idx, valid_idx) in enumerate(cv_strategy.split(X_train, y_train), start=1):
    estimator = clone(xgb_model)

    X_fold_train = X_train.iloc[train_idx]
    y_fold_train = y_train.iloc[train_idx]
    X_valid = X_train.iloc[valid_idx]
    y_valid = y_train.iloc[valid_idx]

    estimator.fit(X_fold_train, y_fold_train)

    y_valid_proba = estimator.predict_proba(X_valid)[:, 1]

    fold_predictions.append(pd.DataFrame({
        "Fold": fold_number,
        "y_valid": y_valid.to_numpy(),
        "y_valid_proba": y_valid_proba,
    }))

xgb_cv_predictions_df = pd.concat(fold_predictions, ignore_index=True)

xgb_cv_predictions_df.head()


,Fold,y_valid,y_valid_proba
0,1,0,0.732687
1,1,1,0.786090
2,1,0,0.033323
3,1,0,0.452017
4,1,0,0.171586


In [6]:
# ==========================================
# 5. Display Selected Cross-Validation Metrics from Out-of-Fold Predictions
# ==========================================

def get_best_f2_threshold(y_true, y_score, beta=2):
    precision_curve, recall_curve, pr_thresholds = precision_recall_curve(y_true, y_score)
    beta_squared = beta ** 2
    numerator = (1 + beta_squared) * precision_curve[:-1] * recall_curve[:-1]
    denominator = beta_squared * precision_curve[:-1] + recall_curve[:-1]
    fbeta_scores = np.divide(
        numerator,
        denominator,
        out=np.zeros_like(numerator),
        where=denominator != 0,
    )
    return pr_thresholds[fbeta_scores.argmax()]


def get_best_tpr_fpr_threshold(y_true, y_score):
    fpr, tpr, roc_thresholds = roc_curve(y_true, y_score)
    finite_threshold_mask = np.isfinite(roc_thresholds)
    fpr = fpr[finite_threshold_mask]
    tpr = tpr[finite_threshold_mask]
    roc_thresholds = roc_thresholds[finite_threshold_mask]
    best_index = (tpr - fpr).argmax()
    return roc_thresholds[best_index], tpr[best_index] - fpr[best_index]


def summarize_threshold(y_true, y_score, threshold, selection_rule):
    y_pred = (y_score >= threshold).astype(int)
    true_negative = ((y_true == 0) & (y_pred == 0)).sum()
    false_positive = ((y_true == 0) & (y_pred == 1)).sum()
    false_positive_rate = false_positive / (false_positive + true_negative)
    true_positive_rate = recall_score(y_true, y_pred, pos_label=1, zero_division=0)
    return {
        "Selection Rule": selection_rule,
        "Threshold": threshold,
        "Validation Recall": true_positive_rate,
        "Validation Precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Validation F1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Validation F2": fbeta_score(y_true, y_pred, beta=2, pos_label=1, zero_division=0),
        "Validation AUPRC": average_precision_score(y_true, y_score),
        "Validation AUROC": roc_auc_score(y_true, y_score),
        "Predicted Positive Rate": y_pred.mean(),
        "TPR - FPR": true_positive_rate - false_positive_rate,
    }


y_valid = xgb_cv_predictions_df["y_valid"]
y_valid_proba = xgb_cv_predictions_df["y_valid_proba"]

default_threshold = 0.5
best_f2_threshold = get_best_f2_threshold(y_valid, y_valid_proba, beta=2)
best_tpr_fpr_threshold, best_tpr_fpr_score = get_best_tpr_fpr_threshold(y_valid, y_valid_proba)

xgb_selected_metrics_df = pd.DataFrame([
    summarize_threshold(y_valid, y_valid_proba, default_threshold, "Default threshold"),
    summarize_threshold(y_valid, y_valid_proba, best_f2_threshold, "Max F2 threshold"),
    summarize_threshold(
        y_valid,
        y_valid_proba,
        best_tpr_fpr_threshold,
        "Max TPR-FPR threshold"
    ),
])

xgb_selected_metrics_df.round(6)


,Selection Rule,Threshold,Validation Recall,Validation Precision,Validation F1,Validation F2,Validation AUPRC,Validation AUROC,Predicted Positive Rate,TPR - FPR
0,Default threshold,0.500000,0.781387,0.315764,0.449772,0.603426,0.432862,0.814934,0.377965,0.476148
1,Max F2 threshold,0.436386,0.839584,0.291025,0.432227,0.609726,0.432862,0.814934,0.440639,0.470864
2,Max TPR-FPR threshold,0.505509,0.776507,0.318319,0.451537,0.602935,0.432862,0.814934,0.372590,0.476733
